# Precompute Graph Embeddings — Google Colab
Runs **Qwen/Qwen3-Embedding-0.6B** on Colab's free T4 GPU to produce all `.npy` embedding caches required by the local pipeline.

**Steps:**
1. Mount Google Drive and set `BASE_PATH` to where you uploaded `temporal_graph_output_v3/`
2. Run all cells — takes ~15–30 min on T4 GPU vs. ~100 h on CPU
3. The `embeddings/` folder will appear inside your Drive path — copy it back locally into `data/jsonls/temporal_graph_output_v3/embeddings/`

> **Runtime**: `Runtime → Change runtime type → T4 GPU`

## 1 · Mount Google Drive

In [5]:
import sys

IN_COLAB = 'google.colab' in sys.modules or 'google.colab' in str(sys.path)
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
    print('Google Drive mounted.')
except ImportError:
    IN_COLAB = False
    print('Not running in Colab — skipping Drive mount.')

Not running in Colab — skipping Drive mount.


## 2 · Configuration
Set `BASE_PATH` to the folder where you uploaded `temporal_graph_output_v3/` in your Drive.

In [ ]:
import re, pathlib, os

# ── ENVIRONMENT ───────────────────────────────────────────────────────────────
# TEST_MODE: set to True for a quick local sanity-check (embeds only 200 items).
# Set to False for the full production run (Colab or local).
TEST_MODE   = False
TEST_LIMIT  = 200   # items per corpus when TEST_MODE is True

# ── PATH CONFIGURATION ────────────────────────────────────────────────────────
if IN_COLAB:
    # ↓ Change this to match where you uploaded the folder in your Drive
    BASE_PATH = '/content/drive/MyDrive/explanability-for-temporal-graphs/explanability-v2/temporal_graph_output_v3'
else:
    # Local path — resolved relative to notebook location
    _nb_dir   = pathlib.Path(os.path.abspath(''))
    _repo_root = _nb_dir.parent if (_nb_dir / '..').resolve().name != _nb_dir.name else _nb_dir
    BASE_PATH = str((_repo_root / 'data' / 'jsonls' / 'temporal_graph_output_v3').resolve())

# ── MODEL ─────────────────────────────────────────────────────────────────────
MODEL_NAME = 'Qwen/Qwen3-Embedding-4B'

# ── BATCH SIZES ───────────────────────────────────────────────────────────────
# T4 GPU: 128 is safe. Local CPU test: keep smaller to stay fast.
MICRO_BATCH = 16 if (TEST_MODE or not IN_COLAB) else 128

# ── DERIVED (do not edit) ─────────────────────────────────────────────────────
def _slug(s):
    s = (s or 'default').strip().lower()
    s = re.sub(r'[^a-z0-9]+', '_', s).strip('_')
    return s or 'default'

MODEL_SLUG    = _slug(MODEL_NAME)
EMBED_BACKEND = 'qwen_server'   # keeps filenames compatible with local pipeline

OUTPUT_DIR = pathlib.Path(BASE_PATH)
EMB_DIR    = OUTPUT_DIR / 'embeddings_4b'
EMB_DIR.mkdir(parents=True, exist_ok=True)

print(f'IN_COLAB    : {IN_COLAB}')
print(f'TEST_MODE   : {TEST_MODE}  (limit={TEST_LIMIT if TEST_MODE else "none"})')
print(f'BASE_PATH   : {BASE_PATH}')
print(f'Exists      : {OUTPUT_DIR.exists()}')
print(f'MODEL_NAME  : {MODEL_NAME}')
print(f'MODEL_SLUG  : {MODEL_SLUG}')
print(f'MICRO_BATCH : {MICRO_BATCH}')
print(f'EMB_DIR     : {EMB_DIR}')

IN_COLAB    : False
TEST_MODE   : True  (limit=200)
BASE_PATH   : C:\Users\ivanr\OneDrive\Documents\New folder (2)\snet\explanability-for-temporal-graphs\data\jsonls\temporal_graph_output_v2
Exists      : True
MODEL_NAME  : Qwen/Qwen3-Embedding-0.6B
MODEL_SLUG  : qwen_qwen3_embedding_0_6b
MICRO_BATCH : 16
EMB_DIR     : C:\Users\ivanr\OneDrive\Documents\New folder (2)\snet\explanability-for-temporal-graphs\data\jsonls\temporal_graph_output_v2\embeddings


## 3 · Install Dependencies & Check GPU

In [ ]:
if IN_COLAB:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'transformers', 'accelerate', 'tqdm', 'psutil'], check=True)

import os
import torch

try:
    import psutil
except Exception:
    psutil = None


def print_runtime_memory(stage=''):
    prefix = f'[{stage}] ' if stage else ''
    if psutil is not None:
        vm = psutil.virtual_memory()
        used_gb = (vm.total - vm.available) / 1e9
        total_gb = vm.total / 1e9
        rss_gb = psutil.Process(os.getpid()).memory_info().rss / 1e9
        print(f'{prefix}RAM used: {used_gb:.2f} / {total_gb:.2f} GB ({vm.percent:.1f}%) | process RSS: {rss_gb:.2f} GB')
    else:
        print(f'{prefix}RAM metrics unavailable (psutil not installed).')

    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'{prefix}GPU alloc/reserved/total: {alloc:.2f}/{reserved:.2f}/{total:.2f} GB')


print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('No GPU - running on CPU (TEST_MODE should be True for local runs)')

print_runtime_memory('startup')

PyTorch : 2.5.1+cpu
CUDA    : False
⚠ No GPU — running on CPU (TEST_MODE should be True for local runs)


## 4 · Load & Verify Uploaded Files

In [ ]:
import json, os, gc

print_runtime_memory('before loading jsonl rows')


def _iter_jsonl(path, limit=None):
    count = 0
    with open(path, encoding='utf-8-sig') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if isinstance(obj, dict):
                    yield obj
                    count += 1
                    if limit and count >= limit:
                        return
            except json.JSONDecodeError:
                pass


def _safe_unique(values, limit=None):
    seen, out = set(), []
    for v in values:
        v = str(v or '').strip()
        if v and v not in seen:
            seen.add(v)
            out.append(v)
            if limit and len(out) >= limit:
                break
    return out


nodes_path = OUTPUT_DIR / 'nodes.jsonl'
edges_path = OUTPUT_DIR / 'edges.jsonl'
tags_path = OUTPUT_DIR / 'tags.jsonl'

for p in [nodes_path, edges_path, tags_path]:
    size_mb = os.path.getsize(p) / 1e6
    print(f'  {p.name:20s}  {size_mb:.1f} MB')

_row_limit = TEST_LIMIT if TEST_MODE else None
print(f'\nLoading rows (limit={_row_limit}) ...')
node_rows = list(_iter_jsonl(nodes_path, limit=_row_limit))
edge_rows = list(_iter_jsonl(edges_path, limit=_row_limit))
tag_rows = list(_iter_jsonl(tags_path, limit=_row_limit))
print(f'  nodes : {len(node_rows):,}')
print(f'  edges : {len(edge_rows):,}')
print(f'  tags  : {len(tag_rows):,}')
print_runtime_memory('after loading jsonl rows')

# -- Lookup maps (key is 'node_uid', not 'uid') -------------------------------
node_label_by_uid = {r['node_uid']: r.get('normalized_label') or r.get('label', '') for r in node_rows}
node_category_by_uid = {r['node_uid']: r.get('category', 'unknown') for r in node_rows}


def node_label(uid):
    return node_label_by_uid.get(uid, uid)


def node_category(uid):
    return node_category_by_uid.get(uid, 'unknown')


# -- Unique text corpora -------------------------------------------------------
node_labels = _safe_unique([r.get('normalized_label') or '' for r in node_rows], limit=_row_limit)
edge_relations = _safe_unique([r.get('relation') or '' for r in edge_rows], limit=_row_limit)
tag_values = _safe_unique([r.get('normalized_tag') or r.get('tag', '') or '' for r in tag_rows], limit=_row_limit)

print(f'\nUnique node labels    : {len(node_labels):,}')
print(f'Unique edge relations : {len(edge_relations):,}')
print(f'Unique tag values     : {len(tag_values):,}')

# Preview first row keys for quick schema check
print(f'\nNode keys  : {list(node_rows[0].keys())}')
print(f'Edge keys  : {list(edge_rows[0].keys())}')
print(f'Tag keys   : {list(tag_rows[0].keys()) if tag_rows else "n/a"}')
print_runtime_memory('after corpus prep')
gc.collect()

  nodes.jsonl           23.7 MB
  edges.jsonl           81.1 MB
  tags.jsonl            5.3 MB

Loading rows (limit=200) ...
  nodes : 200
  edges : 200
  tags  : 200

Unique node labels    : 190
Unique edge relations : 35
Unique tag values     : 200

Node keys  : ['node_uid', 'label', 'category', 'normalized_label', 'source_count', 'source_row_ids']
Edge keys  : ['edge_uid', 'source_uid', 'target_uid', 'relation', 'start', 'end', 'edge_type', 'support_count', 'source_row_ids']
Tag keys   : ['tag_uid', 'tag', 'normalized_tag', 'source_count', 'source_row_ids']


## 5 · Build Retrieval Texts (Edge & Node Sentences)
Same sentence-form logic as the local `precompute_graph_embeddings.py`.

In [ ]:
import gc


def _edge_text(edge_row):
    """Sentence-form edge text - mirrors local precompute_graph_embeddings._edge_text()."""
    source = node_label(edge_row.get('source_uid', ''))
    target = node_label(edge_row.get('target_uid', ''))
    src_cat = node_category(edge_row.get('source_uid', ''))
    tgt_cat = node_category(edge_row.get('target_uid', ''))
    relation = (edge_row.get('relation') or 'related_to').replace('_', ' ')
    start = (edge_row.get('start') or '').split('-')[0]
    end = (edge_row.get('end') or '').split('-')[0]
    if start and end and start != end:
        temporal = f' from {start} to {end}'
    elif start:
        temporal = f' from {start}'
    elif end:
        temporal = f' until {end}'
    else:
        temporal = ''
    return f'{source} [{src_cat}] {relation} {target} [{tgt_cat}]{temporal}'


def _node_text(node_uid):
    """Node text - mirrors local precompute_graph_embeddings._node_text()."""
    label = node_label(node_uid)
    category = node_category(node_uid)
    return f'{label} [{category}]'


print_runtime_memory('before retrieval text build')

# Build ordered UIDs + texts for retrieval artifacts
node_uids = sorted(node_label_by_uid.keys())
node_texts = [_node_text(uid) for uid in node_uids]
print_runtime_memory('after node_texts build')

# edge_uid key is 'edge_uid' in the jsonl schema
edge_uids = [r['edge_uid'] for r in edge_rows]
edge_texts = [_edge_text(r) for r in edge_rows]
print_runtime_memory('after edge_texts build')

print(f'Retrieval node texts : {len(node_texts):,}  e.g. "{node_texts[0]}"')
print(f'Retrieval edge texts : {len(edge_texts):,}  e.g. "{edge_texts[0]}"')

gc.collect()

Retrieval node texts : 200  e.g. "moving assembly line implementation [event]"
Retrieval edge texts : 200  e.g. "1913 [date] within year 1913 [date]"


## 6 · Load Model onto GPU

In [10]:
from transformers import AutoTokenizer, AutoModel
import torch, numpy as np
from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

print(f'Loading {MODEL_NAME} ...')
tokenizer   = AutoTokenizer.from_pretrained(MODEL_NAME)
embed_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
embed_model.eval()
print('Model ready.')

if DEVICE.type == 'cuda':
    print(f'VRAM used after load: {torch.cuda.memory_allocated()/1e9:.2f} GB')

Using device: cpu
Loading Qwen/Qwen3-Embedding-0.6B ...
Model ready.


## 7 · Embedding Helpers (with Checkpointing)

In [ ]:
import pathlib, time


def _mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


def _embed_micro_batch(texts):
    """Embed a single micro-batch of texts, returns (N, dim) float32 numpy array."""
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors='pt')
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        out = embed_model(**inputs)
    pooled = _mean_pool(out.last_hidden_state, inputs['attention_mask'])
    normalized = torch.nn.functional.normalize(pooled, p=2, dim=1)
    return normalized.cpu().numpy().astype(np.float32)


def _cleanup_duplicate_chunks(ckpt_dir):
    """Delete Drive-conflict chunk files like chunk_00000000 (1).npy."""
    removed = 0
    promoted = 0
    for p in sorted(pathlib.Path(ckpt_dir).glob('chunk_*.npy')):
        name = p.name
        if ' (' not in name or not name.endswith(').npy'):
            continue
        base, suffix = name.rsplit(' (', 1)
        count_txt = suffix[:-5]  # drop ').npy'
        if not base.startswith('chunk_') or not count_txt.isdigit():
            continue

        canonical = pathlib.Path(ckpt_dir) / f'{base}.npy'
        try:
            if canonical.exists():
                p.unlink()
                removed += 1
            else:
                p.replace(canonical)
                promoted += 1
        except OSError:
            continue
    return removed, promoted


def _load_checkpoint_chunk(path, expected_rows=None, retries=2):
    """Best-effort loader for checkpoint chunks on flaky synced storage."""
    last_exc = None
    for attempt in range(int(retries) + 1):
        try:
            arr = np.load(path)
            arr = np.asarray(arr, dtype=np.float32)
            if arr.ndim != 2:
                raise ValueError(f'expected 2D array, got shape={arr.shape}')
            if expected_rows is not None and arr.shape[0] != expected_rows:
                raise ValueError(f'row count mismatch: expected {expected_rows}, got {arr.shape[0]}')
            return arr, None
        except (OSError, ValueError, EOFError) as exc:
            last_exc = exc
            if attempt < int(retries):
                time.sleep(0.4 * (attempt + 1))
                continue
            return None, last_exc
    return None, last_exc


def embed_texts(texts, label='Embedding', ckpt_dir=None):
    """
    Embed all texts in micro-batches with tqdm progress bar and optional checkpoint
    resumption. Returns (N, dim) float32 array.
    """
    n = len(texts)
    if n == 0:
        return np.zeros((0, 0), dtype=np.float32)

    results = [None] * n
    done = 0
    ckpt_enabled = ckpt_dir is not None

    if ckpt_enabled:
        ckpt_dir = pathlib.Path(ckpt_dir)
        try:
            # Avoid unnecessary mkdir calls on flaky Drive mounts.
            if not ckpt_dir.exists():
                ckpt_dir.mkdir(parents=True, exist_ok=True)
        except OSError as exc:
            print(f'  WARN checkpoint directory unavailable ({type(exc).__name__}); continuing without checkpointing.')
            ckpt_enabled = False

    if ckpt_enabled:
        removed_dups, promoted_dups = _cleanup_duplicate_chunks(ckpt_dir)
        if removed_dups or promoted_dups:
            print(f'  Checkpoint cleanup {ckpt_dir.name}: removed={removed_dups}, promoted={promoted_dups}')

        # Resume: reload already-computed chunks.
        for i in range(0, n, MICRO_BATCH):
            chunk_path = ckpt_dir / f'chunk_{i:08d}.npy'
            try:
                has_chunk = chunk_path.exists()
            except OSError as exc:
                print(f'  WARN checkpoint storage unavailable ({type(exc).__name__}); disabling checkpointing for this run.')
                ckpt_enabled = False
                break
            if not has_chunk:
                continue

            end = min(i + MICRO_BATCH, n)
            arr, load_err = _load_checkpoint_chunk(chunk_path, expected_rows=end - i)
            if arr is None:
                try:
                    chunk_path.unlink()
                except OSError:
                    pass
                err_name = type(load_err).__name__ if load_err is not None else 'UnknownError'
                print(f'  WARN removed unreadable checkpoint chunk: {chunk_path.name} ({err_name})')
                continue

            results[i:end] = list(arr)
            done += end - i

    bar = tqdm(total=n, initial=done, desc=label, unit='item', dynamic_ncols=True)
    t0 = time.perf_counter()

    for i in range(0, n, MICRO_BATCH):
        end = min(i + MICRO_BATCH, n)
        chunk = texts[i:end]

        # Skip if already loaded from checkpoint.
        if results[i] is not None:
            continue

        arr = _embed_micro_batch(chunk)
        results[i:end] = list(arr)

        if ckpt_enabled:
            try:
                atomic_save(ckpt_dir / f'chunk_{i:08d}.npy', arr)
            except OSError as exc:
                print(f'  WARN checkpoint write failed ({type(exc).__name__}); continuing without checkpointing.')
                ckpt_enabled = False

        bar.update(end - i)

    bar.close()
    elapsed = time.perf_counter() - t0
    actual = n - done  # items actually computed this run
    if actual > 0:
        print(f'  -> {label}: {actual:,} items in {elapsed:.1f}s  ({actual/max(elapsed,1e-6):.1f} items/s)')

    stacked = np.vstack([r if isinstance(r, np.ndarray) else np.array(r) for r in results])
    return stacked.astype(np.float32)


def atomic_save(path, arr):
    tmp = pathlib.Path(str(path) + '.tmp.npy')
    np.save(tmp, arr)
    tmp.replace(path)
    print(f'  Saved {path.name}  shape={arr.shape}')


def write_meta_jsonl(path, rows):
    with open(path, 'w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')


print('Helpers defined.')

Helpers defined.


## 8 · Embed Core Fields (node labels · edge relations · tag values)
These are the primary lookup embeddings used by the query pipeline.

In [ ]:
import gc

if IN_COLAB:
    # Keep checkpoint chunks on local disk to avoid Drive/FUSE endpoint failures.
    CKPT_ROOT_LOCAL = pathlib.Path('/content') / 'temporal_ckpt' / f'checkpoints_{EMBED_BACKEND}_{MODEL_SLUG}'
    CKPT_ROOT_DRIVE = EMB_DIR / f'checkpoints_{EMBED_BACKEND}_{MODEL_SLUG}'
    CKPT_ROOT = CKPT_ROOT_LOCAL
else:
    CKPT_ROOT = EMB_DIR / f'checkpoints_{EMBED_BACKEND}_{MODEL_SLUG}'

CKPT_ROOT.mkdir(parents=True, exist_ok=True)
print(f'CKPT_ROOT active: {CKPT_ROOT}')
if IN_COLAB:
    print(f'CKPT_ROOT local : {CKPT_ROOT_LOCAL}')
    print(f'CKPT_ROOT drive : {CKPT_ROOT_DRIVE}')
print_runtime_memory('core embeddings: start')

# Preflight cleanup for prior interrupted runs.
if CKPT_ROOT.exists():
    total_removed = 0
    total_promoted = 0
    for d in sorted(CKPT_ROOT.iterdir()):
        if not d.is_dir():
            continue
        removed, promoted = _cleanup_duplicate_chunks(d)
        total_removed += removed
        total_promoted += promoted
        if removed or promoted:
            print(f'  Checkpoint cleanup {d.name}: removed={removed}, promoted={promoted}')
    if total_removed or total_promoted:
        print(f'  Checkpoint cleanup total: removed={total_removed}, promoted={total_promoted}')


def _load_existing_matrix(path, expected_rows, label):
    p = pathlib.Path(path)
    if not p.exists():
        return None
    try:
        arr = np.load(p, mmap_mode='r')
        if arr.ndim == 2 and arr.shape[0] == expected_rows:
            print(f'  SKIP {label} - already exists  {p.name}  shape={arr.shape}')
            return np.asarray(arr, dtype=np.float32)
        print(f'  Existing {p.name} has unexpected shape {arr.shape}; recomputing {label}.')
    except Exception as exc:
        print(f'  Existing {p.name} unreadable ({type(exc).__name__}); recomputing {label}.')
    return None


# -- Node normalized_label embeddings -----------------------------------------
print_runtime_memory('core: before node labels')
node_label_npy = EMB_DIR / f'node_normalized_label_embeddings_{EMBED_BACKEND}_{MODEL_SLUG}.npy'
node_label_meta = EMB_DIR / f'node_normalized_label_embeddings_{EMBED_BACKEND}_{MODEL_SLUG}.meta.jsonl'
node_label_matrix = _load_existing_matrix(node_label_npy, len(node_labels), 'Node labels')
if node_label_matrix is None:
    node_label_matrix = embed_texts(
        node_labels,
        label='Node labels',
        ckpt_dir=CKPT_ROOT / 'node_normalized_label',
    )
    atomic_save(node_label_npy, node_label_matrix)
if not node_label_meta.exists():
    write_meta_jsonl(node_label_meta, [{'normalized_label': lbl} for lbl in node_labels])
del node_label_matrix
gc.collect()
print_runtime_memory('core: after node labels')

# -- Edge relation embeddings --------------------------------------------------
print_runtime_memory('core: before edge relations')
edge_relation_npy = EMB_DIR / f'edge_relation_embeddings_{EMBED_BACKEND}_{MODEL_SLUG}.npy'
edge_relation_meta = EMB_DIR / f'edge_relation_embeddings_{EMBED_BACKEND}_{MODEL_SLUG}.meta.jsonl'
edge_relation_matrix = _load_existing_matrix(edge_relation_npy, len(edge_relations), 'Edge relations')
if edge_relation_matrix is None:
    edge_relation_matrix = embed_texts(
        edge_relations,
        label='Edge relations',
        ckpt_dir=CKPT_ROOT / 'edge_relation',
    )
    atomic_save(edge_relation_npy, edge_relation_matrix)
if not edge_relation_meta.exists():
    write_meta_jsonl(edge_relation_meta, [{'relation': r} for r in edge_relations])
del edge_relation_matrix
gc.collect()
print_runtime_memory('core: after edge relations')

# -- Tag normalized_tag embeddings --------------------------------------------
print_runtime_memory('core: before tags')
tag_npy = EMB_DIR / f'tag_normalized_tag_embeddings_{EMBED_BACKEND}_{MODEL_SLUG}.npy'
tag_meta = EMB_DIR / f'tag_normalized_tag_embeddings_{EMBED_BACKEND}_{MODEL_SLUG}.meta.jsonl'
tag_matrix = _load_existing_matrix(tag_npy, len(tag_values), 'Tag values')
if tag_matrix is None:
    tag_matrix = embed_texts(
        tag_values,
        label='Tag values',
        ckpt_dir=CKPT_ROOT / 'tag_normalized_tag',
    )
    atomic_save(tag_npy, tag_matrix)
if not tag_meta.exists():
    write_meta_jsonl(tag_meta, [{'normalized_tag': t} for t in tag_values])
del tag_matrix
gc.collect()
print_runtime_memory('core: after tags')

print('\nCore embeddings complete.')

Node labels:   0%|          | 0/190 [00:00<?, ?item/s]

  → Node labels: 190 items in 20.8s  (9.1 items/s)
  Saved node_normalized_label_embeddings_qwen_server_qwen_qwen3_embedding_0_6b.npy  shape=(190, 1024)


Edge relations:   0%|          | 0/35 [00:00<?, ?item/s]

  → Edge relations: 35 items in 2.6s  (13.6 items/s)
  Saved edge_relation_embeddings_qwen_server_qwen_qwen3_embedding_0_6b.npy  shape=(35, 1024)


Tag values:   0%|          | 0/200 [00:00<?, ?item/s]

  → Tag values: 200 items in 12.5s  (15.9 items/s)
  Saved tag_normalized_tag_embeddings_qwen_server_qwen_qwen3_embedding_0_6b.npy  shape=(200, 1024)

✓ Core embeddings complete.


## 9 · Embed Retrieval Artifacts (full node & edge sentence texts)
These produce the `EdgeSemanticIndex` caches used at query time. This is the largest pass (~354k edges).

In [ ]:
import gc, shutil

print_runtime_memory('retrieval embeddings: start')


def _matrix_file_has_rows(path, expected_rows, label):
    p = pathlib.Path(path)
    if not p.exists():
        return False
    try:
        arr = np.load(p, mmap_mode='r')
        ok = arr.ndim == 2 and arr.shape[0] == expected_rows
        if ok:
            print(f'  SKIP {label} - already exists  {p.name}  shape={arr.shape}')
            return True
        print(f'  Existing {p.name} has unexpected shape {arr.shape}; recomputing {label}.')
    except Exception as exc:
        print(f'  Existing {p.name} unreadable ({type(exc).__name__}); recomputing {label}.')
    return False


def _write_meta_jsonl_iter(path, rows_iter):
    with open(path, 'w', encoding='utf-8') as f:
        for row in rows_iter:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')


def _write_uids_json_from_meta(meta_jsonl_path, out_json_path):
    first = True
    with open(out_json_path, 'w', encoding='utf-8') as w, open(meta_jsonl_path, 'r', encoding='utf-8') as r:
        w.write('[')
        for line in r:
            line = line.strip()
            if not line:
                continue
            try:
                uid = json.loads(line).get('edge_uid', '')
            except Exception:
                uid = ''
            if not first:
                w.write(',')
            w.write(json.dumps(uid, ensure_ascii=False))
            first = False
        w.write(']')


def _safe_chunk_count(ckpt_dir):
    d = pathlib.Path(ckpt_dir)
    try:
        return sum(1 for _ in d.glob('chunk_*.npy'))
    except OSError as exc:
        print(f'  WARN cannot scan checkpoint dir {d} ({type(exc).__name__})')
        return -1


def _chunks_complete(ckpt_dir, total_rows, batch_size):
    # Check exact expected filenames; avoid chunk-content reads here to prevent false negatives
    # on flaky Drive endpoints that would trigger full recompute.
    d = pathlib.Path(ckpt_dir)
    try:
        for i in range(0, total_rows, batch_size):
            p = d / f'chunk_{i:08d}.npy'
            if not p.exists():
                return False
        return True
    except OSError as exc:
        print(f'  WARN cannot validate chunk completeness in {d} ({type(exc).__name__})')
        return False


def _assemble_from_chunks(ckpt_dir, total_rows, batch_size, out_path):
    d = pathlib.Path(ckpt_dir)

    first_chunk = d / 'chunk_00000000.npy'
    first, err = _load_checkpoint_chunk(first_chunk, expected_rows=min(batch_size, total_rows), retries=3)
    if first is None:
        err_name = type(err).__name__ if err is not None else 'UnknownError'
        raise RuntimeError(f'Unable to read first chunk for assembly: {first_chunk} ({err_name})')

    dim = int(first.shape[1])
    mm = np.lib.format.open_memmap(out_path, mode='w+', dtype=np.float32, shape=(total_rows, dim))

    for i in range(0, total_rows, batch_size):
        p = d / f'chunk_{i:08d}.npy'
        expected = min(batch_size, total_rows - i)
        arr, err = _load_checkpoint_chunk(p, expected_rows=expected, retries=3)
        if arr is None:
            err_name = type(err).__name__ if err is not None else 'UnknownError'
            raise RuntimeError(f'Failed to assemble from chunk: {p.name} ({err_name})')
        mm[i:i + expected, :] = arr

    mm.flush()
    del mm
    print(f'  Assembled {out_path.name} from checkpoint chunks (shape=({total_rows}, {dim}))')


# -- Retrieval node embeddings -------------------------------------------------
print_runtime_memory('retrieval: before node embeddings')
node_npy = EMB_DIR / f'node_embeddings_{MODEL_SLUG}.npy'
node_meta_path = EMB_DIR / f'node_embeddings_{MODEL_SLUG}.meta.jsonl'
if not _matrix_file_has_rows(node_npy, len(node_uids), 'Retrieval nodes'):
    if 'node_texts' not in globals() or len(node_texts) != len(node_uids):
        node_texts = [_node_text(uid) for uid in node_uids]
    node_matrix = embed_texts(
        node_texts,
        label='Retrieval nodes',
        ckpt_dir=CKPT_ROOT / 'retrieval_nodes',
    )
    atomic_save(node_npy, node_matrix)
    del node_matrix
    gc.collect()
if not node_meta_path.exists():
    _write_meta_jsonl_iter(
        node_meta_path,
        ({'node_uid': uid, 'label': node_label(uid), 'category': node_category(uid)} for uid in node_uids),
    )
print_runtime_memory('retrieval: after node embeddings')

# -- Retrieval edge embeddings -------------------------------------------------
print_runtime_memory('retrieval: before edge embeddings')
edge_npy = EMB_DIR / f'edge_embeddings_{MODEL_SLUG}.npy'
edge_meta_path = EMB_DIR / f'edge_embeddings_{MODEL_SLUG}.meta.jsonl'
if not _matrix_file_has_rows(edge_npy, len(edge_rows), 'Retrieval edges'):
    if 'edge_texts' not in globals() or len(edge_texts) != len(edge_rows):
        edge_texts = [_edge_text(r) for r in edge_rows]

    # Default: local checkpoint dir.
    edge_ckpt_dir = CKPT_ROOT / 'retrieval_edges'

    # On Colab, prefer Drive checkpoint dir for resume if it has more chunks.
    if IN_COLAB and 'CKPT_ROOT_DRIVE' in globals():
        drive_edge_ckpt_dir = CKPT_ROOT_DRIVE / 'retrieval_edges'
        local_chunks = _safe_chunk_count(edge_ckpt_dir)
        drive_chunks = _safe_chunk_count(drive_edge_ckpt_dir)
        if drive_chunks > local_chunks:
            print(f'  Using Drive checkpoint for Retrieval edges resume: drive={drive_chunks}, local={local_chunks}')
            edge_ckpt_dir = drive_edge_ckpt_dir
        else:
            print(f'  Using local checkpoint for Retrieval edges: local={local_chunks}, drive={drive_chunks}')

    # Fast path: if all chunk filenames exist, assemble final matrix directly on disk (low RAM).
    if _chunks_complete(edge_ckpt_dir, len(edge_rows), MICRO_BATCH):
        print('  All retrieval edge checkpoint chunks present; assembling final matrix from chunks...')
        try:
            _assemble_from_chunks(edge_ckpt_dir, len(edge_rows), MICRO_BATCH, edge_npy)
        except RuntimeError as exc:
            raise RuntimeError(
                f'Checkpoint chunks look complete but assembly failed: {exc}\n'
                'Likely a transient Drive/FUSE issue. Remount Drive and rerun cell 9.'
            )
    else:
        edge_matrix = embed_texts(
            edge_texts,
            label='Retrieval edges',
            ckpt_dir=edge_ckpt_dir,
        )
        atomic_save(edge_npy, edge_matrix)
        del edge_matrix
        gc.collect()
if not edge_meta_path.exists():
    _write_meta_jsonl_iter(
        edge_meta_path,
        ({
            'edge_uid': r.get('edge_uid', str(i)),
            'source_uid': r.get('source_uid', ''),
            'target_uid': r.get('target_uid', ''),
            'relation': r.get('relation', ''),
        } for i, r in enumerate(edge_rows)),
    )
print_runtime_memory('retrieval: after edge embeddings')

# Free large text lists before compatibility copies.
if 'node_texts' in globals():
    del node_texts
if 'edge_texts' in globals():
    del edge_texts
gc.collect()
print_runtime_memory('retrieval: before compat copy')

# -- Compatibility cache (EdgeSemanticIndex reads these directly) -------------
if not edge_npy.exists():
    raise FileNotFoundError(f'Expected retrieval edge matrix not found: {edge_npy}')

for prefix in ('qwen_local', 'qwen_server'):
    compat_npy = EMB_DIR / f'edge_embeddings_{prefix}_{MODEL_SLUG}.npy'
    compat_uids = EMB_DIR / f'edge_embeddings_{prefix}_{MODEL_SLUG}.uids.json'

    if not compat_npy.exists():
        shutil.copyfile(edge_npy, compat_npy)
        print(f'  Copied {edge_npy.name} -> {compat_npy.name}')

    if not compat_uids.exists():
        _write_uids_json_from_meta(edge_meta_path, compat_uids)
        print(f'  Saved {compat_uids.name}')

gc.collect()
print_runtime_memory('retrieval: after compat copy')
print('\nRetrieval embeddings complete.')

Retrieval nodes:   0%|          | 0/200 [00:00<?, ?item/s]

  → Retrieval nodes: 200 items in 25.6s  (7.8 items/s)
  Saved node_embeddings_qwen_qwen3_embedding_0_6b.npy  shape=(200, 1024)


Retrieval edges:   0%|          | 0/200 [00:00<?, ?item/s]

  → Retrieval edges: 200 items in 54.7s  (3.7 items/s)
  Saved edge_embeddings_qwen_qwen3_embedding_0_6b.npy  shape=(200, 1024)
  Saved edge_embeddings_qwen_local_qwen_qwen3_embedding_0_6b.npy  shape=(200, 1024)
  Saved edge_embeddings_qwen_server_qwen_qwen3_embedding_0_6b.npy  shape=(200, 1024)

✓ Retrieval embeddings complete.


## 9a · Recovery Path (compat files only, low RAM)
If retrieval edge embeddings already exist, use this cell in a **fresh runtime** to build only compatibility files.

Run order:
1. Run cell 2 (Configuration)
2. Run this recovery cell

This avoids reloading model/embeddings and minimizes RAM usage.

In [ ]:
import json, pathlib, shutil, re, gc

print_runtime_memory('recovery compat: start')


def _slug_local(s):
    s = (s or 'default').strip().lower()
    s = re.sub(r'[^a-z0-9]+', '_', s).strip('_')
    return s or 'default'


model_slug = MODEL_SLUG if 'MODEL_SLUG' in globals() else _slug_local(MODEL_NAME)
edge_npy = EMB_DIR / f'edge_embeddings_{model_slug}.npy'
edge_meta_path = EMB_DIR / f'edge_embeddings_{model_slug}.meta.jsonl'
edges_path = OUTPUT_DIR / 'edges.jsonl'

if not edge_npy.exists():
    raise FileNotFoundError(
        f'Missing edge embedding matrix: {edge_npy}\n'
        'Run cell 9 first to create retrieval edge embeddings.'
    )

# Create edge meta lazily if missing (streamed, low RAM).
if not edge_meta_path.exists():
    if not edges_path.exists():
        raise FileNotFoundError(f'Missing edges.jsonl: {edges_path}')

    with open(edge_meta_path, 'w', encoding='utf-8') as w, open(edges_path, 'r', encoding='utf-8-sig') as r:
        for i, line in enumerate(r):
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
            except Exception:
                continue
            out = {
                'edge_uid': row.get('edge_uid', str(i)),
                'source_uid': row.get('source_uid', ''),
                'target_uid': row.get('target_uid', ''),
                'relation': row.get('relation', ''),
            }
            w.write(json.dumps(out, ensure_ascii=False) + '\n')
    print(f'  Built {edge_meta_path.name}')


def _write_uids_json_from_meta(meta_jsonl_path, out_json_path):
    first = True
    with open(out_json_path, 'w', encoding='utf-8') as w, open(meta_jsonl_path, 'r', encoding='utf-8') as r:
        w.write('[')
        for line in r:
            line = line.strip()
            if not line:
                continue
            try:
                uid = json.loads(line).get('edge_uid', '')
            except Exception:
                uid = ''
            if not first:
                w.write(',')
            w.write(json.dumps(uid, ensure_ascii=False))
            first = False
        w.write(']')


print_runtime_memory('recovery compat: before copy loop')
for prefix in ('qwen_local', 'qwen_server'):
    compat_npy = EMB_DIR / f'edge_embeddings_{prefix}_{model_slug}.npy'
    compat_uids = EMB_DIR / f'edge_embeddings_{prefix}_{model_slug}.uids.json'

    if not compat_npy.exists():
        shutil.copyfile(edge_npy, compat_npy)
        print(f'  Copied {edge_npy.name} -> {compat_npy.name}')
    else:
        print(f'  SKIP matrix: {compat_npy.name}')

    if not compat_uids.exists():
        _write_uids_json_from_meta(edge_meta_path, compat_uids)
        print(f'  Saved {compat_uids.name}')
    else:
        print(f'  SKIP uids  : {compat_uids.name}')

gc.collect()
print_runtime_memory('recovery compat: done')
print('\nCompatibility recovery complete.')

## 9b · QA Query Embeddings (Backup RAG Index)

Embeds the natural-language **queries** from `qa_index.jsonl`.  
Used as a **fallback**: when graph retrieval returns low-confidence evidence,  
the pipeline cosine-searches these embeddings to find a matching gold record  
and surfaces its `edges` + `gold_answer` directly.

Output files:
- `qa_query_embeddings_<backend>_<model>.npy` — (N, dim) float32 matrix  
- `qa_query_embeddings_<backend>_<model>.uids.json` — record_uid per row

**Skip logic**: if both files already exist on disk the cell is a no-op.  
**Checkpoint resume**: if embedding was interrupted, it restarts from the last saved chunk.


In [ ]:
import gc

qa_npy = EMB_DIR / f'qa_query_embeddings_{EMBED_BACKEND}_{MODEL_SLUG}.npy'
qa_uids_path = EMB_DIR / f'qa_query_embeddings_{EMBED_BACKEND}_{MODEL_SLUG}.uids.json'

print_runtime_memory('qa embeddings: start')

if qa_npy.exists() and qa_uids_path.exists():
    _qa = np.load(qa_npy, mmap_mode='r')
    print(f'SKIP QA query embeddings - already exists  shape={_qa.shape}')
    print_runtime_memory('qa embeddings: already cached')
else:
    # qa_index.jsonl is now generated by the build pipeline alongside
    # nodes.jsonl / edges.jsonl - it always lives in OUTPUT_DIR.
    qa_index_path = OUTPUT_DIR / 'qa_index.jsonl'
    if not qa_index_path.exists():
        raise FileNotFoundError(
            f'qa_index.jsonl not found at {qa_index_path}\n'
            'Re-run the build pipeline locally:\n'
            '  python temporal_graph_processing/build_temporal_graph_output.py \\\n'
            '    --input data/jsonls/temporal_graph.jsonl \\\n'
            '    --output-dir data/jsonls/temporal_graph_output_v3\n'
            'then upload the updated temporal_graph_output_v3/ folder to Drive.'
        )

    qa_records = list(_iter_jsonl(qa_index_path, limit=_row_limit))
    qa_queries = [r.get('query', '') for r in qa_records]
    qa_uids = [r.get('record_uid', str(i)) for i, r in enumerate(qa_records)]
    print_runtime_memory('qa embeddings: after loading qa_index')

    print(f'Embedding {len(qa_queries):,} QA queries...')
    qa_matrix = embed_texts(
        qa_queries,
        label='QA queries',
        ckpt_dir=CKPT_ROOT / 'qa_queries',
    )
    atomic_save(qa_npy, qa_matrix)
    qa_uids_path.write_text(json.dumps(qa_uids, ensure_ascii=False), encoding='utf-8')
    print(f'QA query embeddings complete: {qa_matrix.shape}')
    print(f'  Saved {qa_uids_path.name}')
    del qa_matrix
    gc.collect()
    print_runtime_memory('qa embeddings: done')

## 10 · Summary & Download
List all generated files and optionally zip and download the `embeddings/` folder.

In [ ]:
import numpy as np

print('=' * 65)
print('  EMBEDDING SUMMARY')
print('=' * 65)
total_mb = 0
for p in sorted(EMB_DIR.glob('*.npy')):
    mb = p.stat().st_size / 1e6
    total_mb += mb
    arr = np.load(p, mmap_mode='r')
    shape_str = ' × '.join(str(d) for d in arr.shape)
    print(f'  {p.name:<58s}  {mb:6.1f} MB  {shape_str}')
print('-' * 65)
print(f'  Total: {total_mb:.1f} MB')
print('=' * 65)
print(f'\n✓ All embeddings saved to: {EMB_DIR}')
if not IN_COLAB:
    print('\nLocal test passed! For full production run:')
    print('  1. Set TEST_MODE = False in cell 2')
    print('  2. Upload to Colab with T4 GPU')
else:
    print('\nCopy the embeddings/ folder back locally to:')
    print('  data/jsonls/temporal_graph_output_v3/embeddings/')

  EMBEDDING SUMMARY
  edge_embeddings_qwen_local_qwen_qwen3_embedding_0_6b.npy       0.8 MB  200 × 1024
  edge_embeddings_qwen_qwen3_embedding_0_6b.npy                  0.8 MB  200 × 1024
  edge_embeddings_qwen_server_qwen_qwen3_embedding_0_6b.npy      0.8 MB  200 × 1024
  edge_relation_embeddings_qwen_server_qwen_qwen3_embedding_0_6b.npy     0.1 MB  35 × 1024
  node_embeddings_qwen_qwen3_embedding_0_6b.npy                  0.8 MB  200 × 1024
  node_normalized_label_embeddings_qwen_server_qwen_qwen3_embedding_0_6b.npy     0.8 MB  190 × 1024
  tag_normalized_tag_embeddings_qwen_server_qwen_qwen3_embedding_0_6b.npy     0.8 MB  200 × 1024
-----------------------------------------------------------------
  Total: 5.0 MB

✓ All embeddings saved to: C:\Users\ivanr\OneDrive\Documents\New folder (2)\snet\explanability-for-temporal-graphs\data\jsonls\temporal_graph_output_v2\embeddings

Local test passed! For full production run:
  1. Set TEST_MODE = False in cell 2
  2. Upload to Colab with T4

### Optional: Zip & Download
Run this cell if you want to download the `embeddings/` folder directly to your machine
instead of relying on Drive sync. The zip will be ~500 MB–2 GB depending on graph size.

In [15]:
if IN_COLAB:
    import shutil
    from google.colab import files

    zip_path = '/content/embeddings.zip'
    print(f'Zipping {EMB_DIR} ...')
    shutil.make_archive('/content/embeddings', 'zip', str(EMB_DIR.parent), str(EMB_DIR.name))
    print(f'Zip size: {pathlib.Path(zip_path).stat().st_size / 1e6:.1f} MB')
    print('Downloading...')
    files.download(zip_path)
else:
    print(f'Local run — embeddings already at:\n  {EMB_DIR}')
    print('No download needed.')

Local run — embeddings already at:
  C:\Users\ivanr\OneDrive\Documents\New folder (2)\snet\explanability-for-temporal-graphs\data\jsonls\temporal_graph_output_v2\embeddings
No download needed.
